# 05 - Explorative Datenanalyse: Ölpreise (Yahoo Finance)

**Ziel:** Ölpreisdaten in USD von Yahoo Finance laden, erkunden und erste Qualitätsprüfung durchführen.

**Rohstoffe:** WTI Crude Oil, Brent Crude Oil

**Datenquelle:** Yahoo Finance (via yfinance Library)

---

## 1. Setup und Imports

In [ ]:
# Bibliotheken importieren
import yfinance as yf          # Yahoo Finance API
import pandas as pd            # Datenverarbeitung
import numpy as np             # Numerische Berechnungen
import matplotlib.pyplot as plt # Visualisierung
import seaborn as sns           # Erweiterte Visualisierung
import os                       # Dateipfade

# Darstellung konfigurieren
plt.style.use('seaborn-v0_8')  # Schönerer Plot-Stil
plt.rcParams['figure.figsize'] = (12, 6)  # Standardgrösse für Plots
pd.set_option('display.max_columns', None)  # Alle Spalten anzeigen

print('Setup erfolgreich!')

## 2. Daten laden von Yahoo Finance

Wir laden die Ölpreisdaten für WTI und Brent Crude Oil.
Yahoo Finance verwendet `CL=F` für WTI Crude Oil Futures und `BZ=F` für Brent Crude Oil Futures. Preise sind in USD pro Barrel.

In [2]:
# Konfiguration: Welche Rohstoffe und welcher Zeitraum?
OIL_TICKERS = {
    'CL=F': 'WTI_Crude_Oil',    # WTI Crude Oil Futures (USD/Barrel)
    'BZ=F': 'Brent_Crude_Oil',  # Brent Crude Oil Futures (USD/Barrel)
}

START_DATE = '2022-01-01'
END_DATE = '2026-03-25'

print(f'Zeitraum: {START_DATE} bis {END_DATE}')
print(f'Rohstoffe: {list(OIL_TICKERS.values())}')

Zeitraum: 2022-01-01 bis 2026-03-25
Rohstoffe: ['WTI_Crude_Oil', 'Brent_Crude_Oil']


In [3]:
# Daten laden und in einem Dictionary speichern
oil_data = {}

for yahoo_symbol, oil_name in OIL_TICKERS.items():
    print(f'Lade {oil_name} ({yahoo_symbol})...')
    
    # Daten herunterladen
    ticker = yf.Ticker(yahoo_symbol)
    df = ticker.history(start=START_DATE, end=END_DATE)
    
    # Im Dictionary speichern
    oil_data[oil_name] = df
    
    print(f'  -> {len(df)} Zeilen geladen')

print('\nAlle Daten geladen!')

Lade WTI_Crude_Oil (CL=F)...
  -> 1061 Zeilen geladen
Lade Brent_Crude_Oil (BZ=F)...
  -> 1062 Zeilen geladen

Alle Daten geladen!


## 3. Erste Datenübersicht

Schauen wir uns an, was wir bekommen haben.

In [ ]:
# Übersicht für jeden Rohstoff
for oil_name, df in oil_data.items():
    print(f'\n{"=" * 50}')
    print(f'{oil_name}')
    print(f'{"=" * 50}')
    print(f'Shape (Zeilen x Spalten): {df.shape}')
    print(f'Zeitraum: {df.index.min()} bis {df.index.max()}')
    print(f'Spalten: {list(df.columns)}')
    print(f'Datentypen:\n{df.dtypes}')
    print(f'\nErste 3 Zeilen:')
    display(df.head(3))

## 4. Datenqualitätsprüfung

Wir prüfen:
- Fehlende Werte (NaN)
- Duplikate
- Datentypen
- Statistische Kennzahlen

In [ ]:
# Fehlende Werte prüfen
print('FEHLENDE WERTE PRO ROHSTOFF')
print('=' * 50)

for oil_name, df in oil_data.items():
    missing = df.isnull().sum()
    missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
    
    print(f'\n{oil_name}:')
    if missing.sum() == 0:
        print('  Keine fehlenden Werte!')
    else:
        for col in df.columns:
            if missing[col] > 0:
                print(f'  {col}: {missing[col]} fehlend ({missing_pct[col]}%)')

In [ ]:
# Duplikate prüfen (gleiche Datumswerte)
print('DUPLIKATE PRO ROHSTOFF')
print('=' * 50)

for oil_name, df in oil_data.items():
    dupes = df.index.duplicated().sum()
    print(f'{oil_name}: {dupes} Duplikate')

In [7]:
# Statistische Kennzahlen (Deskriptive Statistik)
for oil_name, df in oil_data.items():
    print(f'\n{"=" * 50}')
    print(f'Deskriptive Statistik: {oil_name}')
    print(f'{"=" * 50}')
    display(df.describe())


Deskriptive Statistik: WTI_Crude_Oil


,Open,High,Low,Close,Volume,Dividends,Stock Splits
count,1061.000000,1061.000000,1061.000000,1061.000000,1.061000e+03,1061.0,1061.0
mean,77.748049,79.137323,76.269274,77.687304,3.098117e+05,0.0,0.0
std,13.306285,13.867048,12.693430,13.278495,1.201023e+05,0.0,0.0
min,55.230000,56.700001,54.980000,55.270000,0.000000e+00,0.0,0.0
25%,68.820000,69.830002,67.699997,68.610001,2.557930e+05,0.0,0.0
50%,75.989998,77.489998,74.769997,75.910004,3.081860e+05,0.0,0.0
75%,83.599998,85.500000,82.389999,83.489998,3.620380e+05,0.0,0.0
max,124.660004,130.500000,120.790001,123.699997,1.107193e+06,0.0,0.0



Deskriptive Statistik: Brent_Crude_Oil


,Open,High,Low,Close,Volume,Dividends,Stock Splits
count,1062.000000,1062.000000,1062.000000,1062.000000,1062.000000,1062.0,1062.0
mean,81.901092,83.261978,80.531102,81.919218,33928.724105,0.0,0.0
std,13.446019,14.045880,12.879407,13.474118,21122.434697,0.0,0.0
min,58.939999,60.410000,58.389999,58.919998,0.000000,0.0,0.0
25%,72.695002,73.677504,71.582500,72.637499,22550.000000,0.0,0.0
50%,80.404999,81.715000,79.340000,80.645000,30431.000000,0.0,0.0
75%,88.027498,89.517498,86.629999,88.057499,39922.000000,0.0,0.0
max,129.570007,137.000000,122.500000,127.980003,235965.000000,0.0,0.0


## 5. Visualisierung

### 5.1 Kursverlauf (Close-Preis in USD/Barrel)

In [ ]:
# Kursverlauf für alle Rohstoffe plotten
fig, axes = plt.subplots(len(oil_data), 1, figsize=(14, 4 * len(oil_data)))

if len(oil_data) == 1:
    axes = [axes]

for ax, (oil_name, df) in zip(axes, oil_data.items()):
    ax.plot(df.index, df['Close'], label=oil_name, linewidth=1)
    ax.set_title(f'Kursverlauf {oil_name} (USD/Barrel)', fontsize=14)
    ax.set_xlabel('Datum')
    ax.set_ylabel('Preis (USD)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Beide Ölpreise überlagert im Vergleich
fig, ax = plt.subplots(figsize=(14, 6))

for oil_name, df in oil_data.items():
    ax.plot(df.index, df['Close'], label=oil_name, linewidth=1)

ax.set_title('Ölpreisvergleich: WTI vs. Brent (USD/Barrel)', fontsize=14)
ax.set_xlabel('Datum')
ax.set_ylabel('Preis (USD)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5.2 Verteilung der täglichen Renditen

In [ ]:
# Tägliche Renditen berechnen und Verteilung anzeigen
fig, axes = plt.subplots(1, len(oil_data), figsize=(6 * len(oil_data), 5))

if len(oil_data) == 1:
    axes = [axes]

for ax, (oil_name, df) in zip(axes, oil_data.items()):
    returns = df['Close'].pct_change().dropna()
    
    ax.hist(returns, bins=50, edgecolor='black', alpha=0.7)
    ax.set_title(f'Tägliche Renditen: {oil_name}', fontsize=12)
    ax.set_xlabel('Rendite (%)')
    ax.set_ylabel('Häufigkeit')
    ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### 5.3 Fehlende Tage identifizieren (Wochenenden, Feiertage)

In [ ]:
# Lücken in den Daten identifizieren
for oil_name, df in oil_data.items():
    print(f'\n{oil_name}:')
    
    date_diffs = pd.Series(df.index).diff()
    
    # Lücken grösser als 3 Tage (Wochenende = 3 Tage normal)
    gaps = date_diffs[date_diffs > pd.Timedelta(days=3)]
    
    if len(gaps) > 0:
        print(f'  {len(gaps)} grössere Lücken gefunden (> 3 Tage):')
        for idx in gaps.index:
            print(f'    {df.index[idx-1].strftime("%Y-%m-%d")} -> {df.index[idx].strftime("%Y-%m-%d")} ({date_diffs[idx].days} Tage)')
    else:
        print('  Keine unerwarteten Lücken gefunden.')

In [12]:
# Alle fehlenden Tage anzeigen (inkl. Wochenenden und Feiertage)
for oil_name, df in oil_data.items():
    print(f'\n{"=" * 50}')
    print(f'{oil_name}')
    print(f'{"=" * 50}')
    
    all_days = pd.date_range(start=df.index.min(), end=df.index.max(), freq='D')
    missing_days = all_days.difference(df.index.normalize())
    
    missing_weekends = [d for d in missing_days if d.weekday() >= 5]
    missing_weekdays = [d for d in missing_days if d.weekday() < 5]
    
    print(f'Gesamte fehlende Tage:     {len(missing_days)}')
    print(f'  davon Wochenenden:       {len(missing_weekends)}')
    print(f'  davon Wochentage:        {len(missing_weekdays)}')
    
    if len(missing_weekdays) > 0:
        print(f'\n  Fehlende Wochentage (Feiertage etc.):')
        for d in missing_weekdays:
            day_name = ['Mo', 'Di', 'Mi', 'Do', 'Fr'][d.weekday()]
            print(f'    {d.strftime("%Y-%m-%d")} ({day_name})')


WTI_Crude_Oil
Gesamte fehlende Tage:     481
  davon Wochenenden:       440
  davon Wochentage:        41

  Fehlende Wochentage (Feiertage etc.):
    2022-01-17 (Mo)
    2022-02-21 (Mo)
    2022-04-15 (Fr)
    2022-05-30 (Mo)
    2022-06-20 (Mo)
    2022-07-04 (Mo)
    2022-09-05 (Mo)
    2022-11-24 (Do)
    2022-12-26 (Mo)
    2023-01-02 (Mo)
    2023-01-16 (Mo)
    2023-02-20 (Mo)
    2023-04-07 (Fr)
    2023-05-29 (Mo)
    2023-06-19 (Mo)
    2023-07-04 (Di)
    2023-09-04 (Mo)
    2023-11-23 (Do)
    2023-12-25 (Mo)
    2024-01-01 (Mo)
    2024-01-15 (Mo)
    2024-02-19 (Mo)
    2024-03-29 (Fr)
    2024-05-27 (Mo)
    2024-06-19 (Mi)
    2024-07-04 (Do)
    2024-09-02 (Mo)
    2024-11-28 (Do)
    2024-12-25 (Mi)
    2025-01-01 (Mi)
    2025-01-20 (Mo)
    2025-02-17 (Mo)
    2025-04-18 (Fr)
    2025-05-26 (Mo)
    2025-06-19 (Do)
    2025-09-01 (Mo)
    2025-11-27 (Do)
    2025-12-25 (Do)
    2026-01-01 (Do)
    2026-01-19 (Mo)
    2026-02-16 (Mo)

Brent_Crude_Oil
Gesamte fehlend

## 6. Rohdaten speichern

Die Rohdaten werden unverändert als CSV gespeichert.

In [13]:
# Rohdaten als CSV speichern
OUTPUT_DIR = '../../data/raw/oil/yahoo'
os.makedirs(OUTPUT_DIR, exist_ok=True)

for oil_name, df in oil_data.items():
    filename = f'{oil_name}_{START_DATE}_to_{END_DATE}.csv'
    filepath = os.path.join(OUTPUT_DIR, filename)
    
    df.to_csv(filepath)
    print(f'Gespeichert: {filepath} ({len(df)} Zeilen)')

print('\nAlle Rohdaten gespeichert!')

Gespeichert: ../../data/raw/oil/yahoo/WTI_Crude_Oil_2022-01-01_to_2026-03-25.csv (1061 Zeilen)
Gespeichert: ../../data/raw/oil/yahoo/Brent_Crude_Oil_2022-01-01_to_2026-03-25.csv (1062 Zeilen)

Alle Rohdaten gespeichert!


## 7. Zusammenfassung

### Erkenntnisse aus der EDA:
- **Datenumfang:** (hier Ergebnisse eintragen nach Ausführung)
- **Fehlende Werte:** (hier Ergebnisse eintragen)
- **Duplikate:** (hier Ergebnisse eintragen)
- **Auffälligkeiten:** (hier Ergebnisse eintragen)

### Nächste Schritte:
1. Ölpreisdaten bereinigen und normalisieren
2. Korrelation mit Forex-Daten untersuchen
3. Öl als möglichen Einflussfaktor auf Währungskurse analysieren